# Experiment 4: Evaluation of Sensor Reusability and Failure

## Overview
Evaluate the sensor's lifespan, reusability limits, and identify failure patterns. Determine usage cycles and signal degradation thresholds.

**Protocol Alignment**: This notebook implements Experiment 4 from the Official Testing Protocol for Fiber Optics SERS Sensor (Turkey Rinsate).

## Objective
Determine usage cycles and signal degradation thresholds for sensor reusability assessment.

## Key Concepts
- **Sensor reusability**: Number of times a sensor can be used before degradation
- **Failure detection**: Identifying when sensors fail or degrade beyond acceptable thresholds
- **Signal degradation**: Tracking how sensor signals change over repeated uses
- **Usage cycles**: Quantifying the number of measurements before failure
- **Degradation thresholds**: Defining acceptable signal quality limits

## Protocol Requirements
1. ⏳ Track sensor usage over multiple measurement cycles
2. ⏳ Monitor signal quality degradation over time
3. ⏳ Identify failure patterns and degradation thresholds
4. ⏳ Calculate average usage cycles before failure
5. ⏳ Assess reusability limits and failure rates

## Configuration
- **Sample Type**: Turkey rinsate from Cargill
- **Target Pathogen**: Salmonella (multiple serovars)
- **Failure Criteria**: Signal quality thresholds to be defined

## Status
🚧 **Not Yet Implemented** - This notebook is a placeholder for future implementation.


In [ ]:
# Imports and Setup
# --------------------------------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Dict
import warnings

# Project-specific imports
from sensd_sers_analysis.data import load_dataset_by_serotypes

warnings.filterwarnings("ignore")

# Plotting style
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ All imports completed successfully")

## 1. Data Loading and Sensor Tracking Setup

Load SERS data and organize by sensor_id to track usage over time.

In [ ]:
# Load SERS Data for Reusability Analysis
# --------------------------------------------------------------------------------------------------

# Define dataset location
data_folder = "../example_data/SERS Data 8 (April 2025)/"
signals_folders = ["March testing-Dilutions"]  # Adjust based on available data

# Load all serotypes to track sensor usage across different samples
serotypes = ["ST", "SE", "Mix"]  # Use all available serotypes

try:
    # Load data
    data_df = load_dataset_by_serotypes(data_folder, signals_folders, serotypes)
    print(f"✅ Successfully loaded {len(data_df)} data entries")

    if not data_df.empty:
        print("\n📊 Dataset Structure Analysis:")
        print("=" * 60)
        print(f"Number of data entries: {len(data_df)}")

        # Sensor distribution
        sensor_counts = data_df["sensor_id"].value_counts()
        print("\nSensor distribution:")
        print(f"  Total unique sensors: {len(sensor_counts)}")
        print("  Sensors with most measurements:")
        for sensor_id, count in sensor_counts.head(10).items():
            print(f"    {sensor_id}: {count} measurements")

        # Test ID distribution (to understand measurement sequence)
        print("\nTest ID range per sensor (sample of 5 sensors):")
        sample_sensors = sensor_counts.head(5).index
        for sensor_id in sample_sensors:
            sensor_data = data_df[data_df["sensor_id"] == sensor_id]
            test_ids = sensor_data["test_id"].unique()
            print(
                f"  {sensor_id}: {len(test_ids)} tests - {sorted([str(t) for t in test_ids])[:5]}..."
            )

except Exception as e:
    print(f"❌ Error loading data: {e}")
    import traceback

    traceback.print_exc()
    data_df = pd.DataFrame()

## 2. Extract Test Number and Sort by Measurement Sequence

Extract numeric test numbers to establish measurement order (treating each test as a sequential measurement).

In [ ]:
# Extract Test Number and Create Measurement Sequence
# --------------------------------------------------------------------------------------------------


def extract_test_number(test_id: str) -> int:
    """Extract numeric part from Test ID like 'Test12' -> 12."""
    import re

    m = re.search(r"(\d+)", str(test_id))
    return int(m.group(1)) if m else -1


if not data_df.empty:
    # Extract test numbers
    data_df["test_number"] = data_df["test_id"].apply(extract_test_number)

    # Create measurement sequence: sort by sensor_id, then test_number, then signal_index
    data_df = data_df.sort_values(["sensor_id", "test_number", "signal_index"]).reset_index(
        drop=True
    )

    # Create a sequential measurement index per sensor
    data_df["measurement_index"] = data_df.groupby("sensor_id").cumcount() + 1

    print("✅ Measurement sequence created")
    print("\nSample measurement sequence for first sensor:")
    first_sensor = data_df["sensor_id"].iloc[0]
    sample_data = data_df[data_df["sensor_id"] == first_sensor].head(10)
    print(
        sample_data[
            ["sensor_id", "test_id", "test_number", "measurement_index", "concentration"]
        ].to_string()
    )

## 3. Calculate Signal Quality Metrics Over Time

Track signal quality metrics (peak intensity, AUC, signal-to-noise ratio) over measurement cycles.

In [ ]:
# Calculate Signal Quality Metrics
# --------------------------------------------------------------------------------------------------


def calculate_signal_metrics(signal: np.ndarray) -> Dict[str, float]:
    """
    Calculate signal quality metrics.

    Returns:
        Dictionary with peak_intensity, auc, snr, mean_intensity
    """
    peak_intensity = np.max(signal)
    auc = np.trapz(signal)  # Area under curve
    mean_intensity = np.mean(signal)
    std_intensity = np.std(signal)
    snr = mean_intensity / (std_intensity + 1e-10)  # Signal-to-noise ratio

    return {
        "peak_intensity": peak_intensity,
        "auc": auc,
        "mean_intensity": mean_intensity,
        "snr": snr,
    }


if not data_df.empty:
    # Calculate metrics for each signal
    metrics_list = []
    for idx, row in data_df.iterrows():
        metrics = calculate_signal_metrics(row["signal"])
        metrics["sensor_id"] = row["sensor_id"]
        metrics["measurement_index"] = row["measurement_index"]
        metrics["test_number"] = row["test_number"]
        metrics["concentration"] = row["concentration"]
        metrics_list.append(metrics)

    metrics_df = pd.DataFrame(metrics_list)

    print("✅ Signal quality metrics calculated")
    print("\nSample metrics:")
    print(metrics_df.head(10).to_string())

## 4. Track Sensor Degradation Over Time

Plot signal quality metrics over measurement cycles to identify degradation patterns.

In [ ]:
# Track Sensor Degradation Over Time
# --------------------------------------------------------------------------------------------------

if not metrics_df.empty:
    # Select sensors with sufficient measurements for analysis
    sensor_measurement_counts = metrics_df.groupby("sensor_id")["measurement_index"].max()
    sensors_with_enough_data = sensor_measurement_counts[sensor_measurement_counts >= 5].index

    print(f"\n📊 Sensors with ≥5 measurements: {len(sensors_with_enough_data)}")

    if len(sensors_with_enough_data) > 0:
        # Plot degradation for multiple sensors
        n_sensors_to_plot = min(10, len(sensors_with_enough_data))
        selected_sensors = sensors_with_enough_data[:n_sensors_to_plot]

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle("Sensor Reusability and Degradation Analysis", fontsize=16, fontweight="bold")

        # Plot 1: Peak Intensity over time
        ax1 = axes[0, 0]
        for sensor_id in selected_sensors:
            sensor_metrics = metrics_df[metrics_df["sensor_id"] == sensor_id].sort_values(
                "measurement_index"
            )
            ax1.plot(
                sensor_metrics["measurement_index"],
                sensor_metrics["peak_intensity"],
                marker="o",
                markersize=4,
                alpha=0.7,
                label=sensor_id,
                linewidth=1.5,
            )
        ax1.set_xlabel("Measurement Index (Usage Cycle)", fontsize=12)
        ax1.set_ylabel("Peak Intensity", fontsize=12)
        ax1.set_title("Peak Intensity Over Time", fontsize=13, fontweight="bold")
        ax1.legend(fontsize=8, ncol=2)
        ax1.grid(True, alpha=0.3)

        # Plot 2: AUC over time
        ax2 = axes[0, 1]
        for sensor_id in selected_sensors:
            sensor_metrics = metrics_df[metrics_df["sensor_id"] == sensor_id].sort_values(
                "measurement_index"
            )
            ax2.plot(
                sensor_metrics["measurement_index"],
                sensor_metrics["auc"],
                marker="o",
                markersize=4,
                alpha=0.7,
                label=sensor_id,
                linewidth=1.5,
            )
        ax2.set_xlabel("Measurement Index (Usage Cycle)", fontsize=12)
        ax2.set_ylabel("Area Under Curve (AUC)", fontsize=12)
        ax2.set_title("AUC Over Time", fontsize=13, fontweight="bold")
        ax2.legend(fontsize=8, ncol=2)
        ax2.grid(True, alpha=0.3)

        # Plot 3: SNR over time
        ax3 = axes[1, 0]
        for sensor_id in selected_sensors:
            sensor_metrics = metrics_df[metrics_df["sensor_id"] == sensor_id].sort_values(
                "measurement_index"
            )
            ax3.plot(
                sensor_metrics["measurement_index"],
                sensor_metrics["snr"],
                marker="o",
                markersize=4,
                alpha=0.7,
                label=sensor_id,
                linewidth=1.5,
            )
        ax3.set_xlabel("Measurement Index (Usage Cycle)", fontsize=12)
        ax3.set_ylabel("Signal-to-Noise Ratio (SNR)", fontsize=12)
        ax3.set_title("SNR Over Time", fontsize=13, fontweight="bold")
        ax3.legend(fontsize=8, ncol=2)
        ax3.grid(True, alpha=0.3)

        # Plot 4: Average degradation trend (all sensors)
        ax4 = axes[1, 1]
        max_measurements = metrics_df["measurement_index"].max()
        avg_peak_by_cycle = []
        std_peak_by_cycle = []

        for cycle in range(1, min(max_measurements + 1, 50)):  # Limit to first 50 cycles
            cycle_data = metrics_df[metrics_df["measurement_index"] == cycle]["peak_intensity"]
            if len(cycle_data) > 0:
                avg_peak_by_cycle.append(cycle_data.mean())
                std_peak_by_cycle.append(cycle_data.std())
            else:
                break

        if avg_peak_by_cycle:
            cycles = range(1, len(avg_peak_by_cycle) + 1)
            ax4.plot(cycles, avg_peak_by_cycle, marker="o", linewidth=2, markersize=6, color="red")
            ax4.fill_between(
                cycles,
                np.array(avg_peak_by_cycle) - np.array(std_peak_by_cycle),
                np.array(avg_peak_by_cycle) + np.array(std_peak_by_cycle),
                alpha=0.3,
                color="red",
            )
            ax4.set_xlabel("Measurement Index (Usage Cycle)", fontsize=12)
            ax4.set_ylabel("Average Peak Intensity", fontsize=12)
            ax4.set_title("Average Degradation Trend (All Sensors)", fontsize=13, fontweight="bold")
            ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print("✅ Degradation analysis completed")
    else:
        print("⚠️ Not enough sensors with sufficient measurements for degradation analysis")

## 5. Failure Detection and Reusability Assessment

Identify when sensors fail based on signal quality thresholds and calculate usage cycles before failure.

In [ ]:
# Failure Detection and Reusability Assessment
# --------------------------------------------------------------------------------------------------

if not metrics_df.empty and len(sensors_with_enough_data) > 0:
    # Define failure thresholds (using percentiles of initial measurements)
    initial_measurements = metrics_df[metrics_df["measurement_index"] <= 3]  # First 3 measurements
    if len(initial_measurements) > 0:
        baseline_peak = initial_measurements["peak_intensity"].median()
        baseline_snr = initial_measurements["snr"].median()

        # Failure threshold: 50% degradation from baseline
        peak_threshold = baseline_peak * 0.5
        snr_threshold = baseline_snr * 0.5

        print("\n📊 Failure Thresholds:")
        print(f"Baseline Peak Intensity: {baseline_peak:.2f}")
        print(f"Peak Failure Threshold (50% degradation): {peak_threshold:.2f}")
        print(f"Baseline SNR: {baseline_snr:.2f}")
        print(f"SNR Failure Threshold (50% degradation): {snr_threshold:.2f}")

        # Detect failures per sensor
        failure_analysis = []
        for sensor_id in sensors_with_enough_data:
            sensor_metrics = metrics_df[metrics_df["sensor_id"] == sensor_id].sort_values(
                "measurement_index"
            )

            # Find first failure point
            peak_failures = sensor_metrics[sensor_metrics["peak_intensity"] < peak_threshold]
            snr_failures = sensor_metrics[sensor_metrics["snr"] < snr_threshold]

            first_peak_failure = (
                peak_failures["measurement_index"].min() if len(peak_failures) > 0 else None
            )
            first_snr_failure = (
                snr_failures["measurement_index"].min() if len(snr_failures) > 0 else None
            )

            # Use the earlier failure
            first_failure = None
            if first_peak_failure is not None and first_snr_failure is not None:
                first_failure = min(first_peak_failure, first_snr_failure)
            elif first_peak_failure is not None:
                first_failure = first_peak_failure
            elif first_snr_failure is not None:
                first_failure = first_snr_failure

            max_measurements = sensor_metrics["measurement_index"].max()
            final_peak = sensor_metrics["peak_intensity"].iloc[-1]
            final_snr = sensor_metrics["snr"].iloc[-1]

            failure_analysis.append(
                {
                    "sensor_id": sensor_id,
                    "total_measurements": max_measurements,
                    "measurements_before_failure": first_failure
                    if first_failure
                    else max_measurements,
                    "failed": first_failure is not None,
                    "final_peak_intensity": final_peak,
                    "final_snr": final_snr,
                    "peak_degradation_pct": ((baseline_peak - final_peak) / baseline_peak * 100)
                    if baseline_peak > 0
                    else 0,
                }
            )

        failure_df = pd.DataFrame(failure_analysis)

        # Summary statistics
        print("\n📊 Reusability Summary:")
        print(f"Total sensors analyzed: {len(failure_df)}")
        print(f"Sensors that failed: {failure_df['failed'].sum()}")
        print(f"Sensors still functional: {(~failure_df['failed']).sum()}")

        if failure_df["failed"].sum() > 0:
            failed_sensors = failure_df[failure_df["failed"]]
            print(
                f"\nAverage measurements before failure: {failed_sensors['measurements_before_failure'].mean():.1f}"
            )
            print(
                f"Median measurements before failure: {failed_sensors['measurements_before_failure'].median():.1f}"
            )
            print(
                f"Min measurements before failure: {failed_sensors['measurements_before_failure'].min():.0f}"
            )
            print(
                f"Max measurements before failure: {failed_sensors['measurements_before_failure'].max():.0f}"
            )

        # Visualization
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Plot 1: Distribution of measurements before failure
        ax1 = axes[0]
        if failure_df["failed"].sum() > 0:
            failed_sensors = failure_df[failure_df["failed"]]
            ax1.hist(
                failed_sensors["measurements_before_failure"], bins=20, edgecolor="black", alpha=0.7
            )
            ax1.axvline(
                failed_sensors["measurements_before_failure"].mean(),
                color="red",
                linestyle="--",
                linewidth=2,
                label=f"Mean: {failed_sensors['measurements_before_failure'].mean():.1f}",
            )
            ax1.set_xlabel("Measurements Before Failure", fontsize=12)
            ax1.set_ylabel("Number of Sensors", fontsize=12)
            ax1.set_title("Distribution of Sensor Lifespan", fontsize=13, fontweight="bold")
            ax1.legend()
            ax1.grid(True, alpha=0.3)
        else:
            ax1.text(
                0.5,
                0.5,
                "No sensor failures detected\nwith current thresholds",
                ha="center",
                va="center",
                transform=ax1.transAxes,
                fontsize=12,
            )
            ax1.set_title("Distribution of Sensor Lifespan", fontsize=13, fontweight="bold")

        # Plot 2: Failure rate over time
        ax2 = axes[1]
        if failure_df["failed"].sum() > 0:
            max_cycle = int(failure_df["measurements_before_failure"].max())
            failure_rate_by_cycle = []
            for cycle in range(1, max_cycle + 1):
                failures_by_cycle = (failure_df["measurements_before_failure"] <= cycle).sum()
                failure_rate = failures_by_cycle / len(failure_df) * 100
                failure_rate_by_cycle.append(failure_rate)

            ax2.plot(
                range(1, max_cycle + 1),
                failure_rate_by_cycle,
                marker="o",
                linewidth=2,
                markersize=6,
            )
            ax2.set_xlabel("Measurement Index (Usage Cycle)", fontsize=12)
            ax2.set_ylabel("Cumulative Failure Rate (%)", fontsize=12)
            ax2.set_title("Sensor Failure Rate Over Time", fontsize=13, fontweight="bold")
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim([0, 105])
        else:
            ax2.text(
                0.5,
                0.5,
                "No sensor failures detected\nwith current thresholds",
                ha="center",
                va="center",
                transform=ax2.transAxes,
                fontsize=12,
            )
            ax2.set_title("Sensor Failure Rate Over Time", fontsize=13, fontweight="bold")

        plt.tight_layout()
        plt.show()

        print("\n✅ Failure detection and reusability assessment completed")
        print("\nDetailed failure analysis:")
        print(failure_df.to_string())